In [ ]:
!pip install pandas
!pip install datasets
!pip install pycfg
!pip install transformers
!pip install astunparse
# !pip install pygraphviz


In [ ]:
import pandas as pd
from copy import deepcopy
import ast
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainer, Seq2SeqTrainingArguments
from transformers import DataCollatorForSeq2Seq
from pycfg.pycfg import PyCFG, CFGNode

In [ ]:
df = pd.read_json("Datasets/train.jsonl", lines=True)

In [ ]:
df.head()

In [ ]:
df.shape[0]

In [ ]:
# check the duplicated data
hashed_df = deepcopy(df)
hashed_df['version_data'] = hashed_df['version_data'].map(lambda x: str(x) if isinstance(x, list) else x)


In [ ]:
print(f"Number of duplicated rows: {hashed_df.duplicated().sum()}")

In [ ]:
check_null_df = deepcopy(df)
check_null_df = check_null_df.isnull().sum().sum()
check_null_df

In [ ]:
#--------------NBot required
def infer_return_type(code_str: str) -> str:
    try:
        tree = ast.parse(code_str)
        for node in ast.walk(tree):
            if isinstance(node, ast.Return):
                value = node.value
                if isinstance(value, ast.Constant):  # Python 3.8+
                    return type(value.value).__name__
                elif isinstance(value, ast.List):
                    return "list"
                elif isinstance(value, ast.Dict):
                    return "dict"
                elif isinstance(value, ast.Tuple):
                    return "tuple"
                elif isinstance(value, ast.Call):
                    return "function call"
                elif isinstance(value, ast.Name):
                    return "variable"
                else:
                    return value.__class__.__name__
    except Exception as e:
        return f"Error: {e}"
    return "None"

In [ ]:
# docstring partitioning ------------ not needed

for each_row in df["version_data"]:
    commit_time = ""
    for each_version in each_row:
        if commit_time:
            if commit_time < each_version["commit_date_time"]:
                commit_time = each_version["commit_date_time"]
        else:
            commit_time = each_version["commit_date_time"]
      
    introduction = "" 
    params = ""
    return_type_list = ["list", "tuple", "dict", "set", "frozenset", "int", "str", "float", "bool", None]
    return_value = ""
    
    for each_version in each_row:
        if each_version["commit_date_time"] == commit_time:
            code = each_version["code"]
            return_type = infer_return_type(code)
            print(return_type)
            
            

In [ ]:
# Generate new dataframe
updated_df = pd.DataFrame(columns=['Code', 'Docstring', 'Parameters', 'Semantics'])

for each_row in df["version_data"]:
    new_row_data = {}

    commit_time = ""
    for each_version in each_row:
        if commit_time:
            if commit_time < each_version["commit_date_time"]:
                commit_time = each_version["commit_date_time"]
        else:
            commit_time = each_version["commit_date_time"]

    for each_version in each_row:
        if each_version["commit_date_time"] == commit_time:
            dataset_code = each_version["code"]
            print(dataset_code)
            new_row_data["Code"] = dataset_code
            new_row_data["Docstring"] = each_version["docstring"]
            new_row_data["Parameters"] = dataset_code[dataset_code.find("(") : dataset_code.find(")") + 1]
            new_row_data["Semantics"] = None
            
            new_row_df = pd.DataFrame([new_row_data])
            updated_df = pd.concat([updated_df, new_row_df], ignore_index=True)
print(updated_df)


# AST Embeddings


In [ ]:
!pip install torch

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ASTEncoder(nn.Module):
    def __init__(self, d_model, n_heads, rel_pos_emb_dim):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.rel_pos_emb = nn.Embedding(512, rel_pos_emb_dim)  # max distance 512
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.fc = nn.Linear(d_model + rel_pos_emb_dim, d_model)

    def forward(self, F_ast, sibling_matrix, ancestor_matrix, distance_matrix):
        """
        F_ast: [batch, n_nodes, d_model] - node features from preorder traversal
        sibling_matrix: [batch, n_nodes, n_nodes] - binary mask for siblings
        ancestor_matrix: [batch, n_nodes, n_nodes] - binary mask for ancestor-descendant
        distance_matrix: [batch, n_nodes, n_nodes] - int distances for rel pos emb
        """
        # Get relative positional embeddings
        rel_pos_emb = self.rel_pos_emb(distance_matrix)  # [batch, n_nodes, n_nodes, rel_pos_emb_dim]

        # Aggregate strong relationships (siblings + ancestor-descendant)
        strong_rel_mask = (sibling_matrix | ancestor_matrix).bool()  # [batch, n_nodes, n_nodes]

        # Decoupled attention: mask attention to only strong relationships
        attn_mask = ~strong_rel_mask  # invert for attn mask (True = block)
        # attn_mask should be [batch, n_nodes, n_nodes] for batch_first=True

        # Fix: expand attn_mask for MultiheadAttention
        batch_size, n_nodes, _ = attn_mask.shape
        attn_mask = attn_mask.repeat(self.n_heads, 1, 1)  # [num_heads * batch, n_nodes, n_nodes]

        # Multihead attention (self-attention)
        attn_output, _ = self.attn(F_ast, F_ast, F_ast, attn_mask=attn_mask)

        # Concatenate with relative positional embeddings (mean over neighbors)
        rel_pos_emb_mean = (rel_pos_emb * strong_rel_mask.unsqueeze(-1)).sum(2) / (strong_rel_mask.sum(2, keepdim=True) + 1e-6)
        out = torch.cat([attn_output, rel_pos_emb_mean], dim=-1)
        out = self.fc(out)
        return out

In [ ]:
import torch
import ast
import pandas as pd

# Ensure ASTEncoder is defined in the notebook before running this cell

def process_row(code, encoder_class, rel_pos_emb_dim=8, n_heads=2):
    if not isinstance(code, str) or not code.strip():
        return None
    try:
        tree = ast.parse(code)
        preorder_nodes = []
        def preorder_traversal(node, parent_idx=None, siblings=None):
            idx = len(preorder_nodes)
            preorder_nodes.append((node, parent_idx, siblings))
            children = list(ast.iter_child_nodes(node))
            for i, child in enumerate(children):
                preorder_traversal(child, idx, [idx + j + 1 for j in range(len(children)) if j != i])
        preorder_traversal(tree)
        n_nodes = len(preorder_nodes)
        node_types = [type(n[0]).__name__ for n in preorder_nodes]
        unique_types = {t: i for i, t in enumerate(set(node_types))}
        d_model = len(unique_types)
        # Pad d_model to be divisible by n_heads
        if d_model % n_heads != 0:
            new_d_model = ((d_model // n_heads) + 1) * n_heads
        else:
            new_d_model = d_model
        F_ast = torch.zeros(1, n_nodes, new_d_model)
        for i, t in enumerate(node_types):
            F_ast[0, i, unique_types[t]] = 1
        sibling_matrix = torch.zeros(1, n_nodes, n_nodes, dtype=torch.bool)
        ancestor_matrix = torch.zeros(1, n_nodes, n_nodes, dtype=torch.bool)
        distance_matrix = torch.zeros(1, n_nodes, n_nodes, dtype=torch.long)
        for i, (_, _, siblings) in enumerate(preorder_nodes):
            if siblings:
                for sib in siblings:
                    sibling_matrix[0, i, sib] = 1
        def fill_ancestors(idx, parent_idx, depth):
            if parent_idx is not None:
                ancestor_matrix[0, parent_idx, idx] = 1
                distance_matrix[0, parent_idx, idx] = depth
                fill_ancestors(parent_idx, preorder_nodes[parent_idx][1], depth + 1)
        for idx, (_, parent_idx, _) in enumerate(preorder_nodes):
            fill_ancestors(idx, parent_idx, 1)
        encoder = encoder_class(d_model=new_d_model, n_heads=n_heads, rel_pos_emb_dim=rel_pos_emb_dim)
        with torch.no_grad():
            output = encoder(F_ast, sibling_matrix, ancestor_matrix, distance_matrix)
        return {
            'sibling_matrix': sibling_matrix.cpu().numpy(),
            'ancestor_matrix': ancestor_matrix.cpu().numpy(),
            'distance_matrix': distance_matrix.cpu().numpy(),
            'encoder_output': output.cpu().numpy(),
            'node_types': node_types
        }
    except Exception as e:
        return {'error': str(e)}

# Apply to the entire DataFrame (updated_df)
results = updated_df['Code'].apply(lambda code: process_row(code, ASTEncoder))

# Save outputs to new columns
updated_df['sibling_matrix'] = results.apply(lambda x: x.get('sibling_matrix') if isinstance(x, dict) else None)
updated_df['ancestor_matrix'] = results.apply(lambda x: x.get('ancestor_matrix') if isinstance(x, dict) else None)
updated_df['distance_matrix'] = results.apply(lambda x: x.get('distance_matrix') if isinstance(x, dict) else None)
updated_df['encoder_output'] = results.apply(lambda x: x.get('encoder_output') if isinstance(x, dict) else None)
updated_df['node_types'] = results.apply(lambda x: x.get('node_types') if isinstance(x, dict) else None)
updated_df['ast_error'] = results.apply(lambda x: x.get('error') if isinstance(x, dict) and 'error' in x else None)

# Optionally, save to CSV or inspect
# updated_df.to_pickle('updated_df_with_ast_outputs.pkl')
updated_df.head()

In [ ]:
# Extract Metadata
import ast

def extract_function_metadata(code: str):
    try:
        tree = ast.parse(code)
        func_node = next((n for n in ast.walk(tree) if isinstance(n, ast.FunctionDef)), None)
        if func_node:
            params = [arg.arg for arg in func_node.args.args]
            return_types = ast.unparse(func_node.returns) if func_node.returns else None
            return {
                "name": func_node.name,
                "params": params,
                "return_type": return_types #----------
            }
           
    except:
        return 
    
updated_df["metadata"] = updated_df["Code"].apply(extract_function_metadata)
print(updated_df)

# CFG Model


In [ ]:
updated_df.shape

In [ ]:
import ast
import networkx as nx
from typing import List, Dict, Any

class CFGNode:
    """Class representing each node in the Control Flow Graph (CFG)"""
    def __init__(self, node_id: int, statement: str, node_type: str = "statement"):
        self.id = node_id
        self.statement = statement
        self.node_type = node_type
        self.successors = []
        self.predecessors = []
    
    def add_successor(self, successor_node):
        """Add a successor node"""
        if successor_node not in self.successors:
            self.successors.append(successor_node)
        if self not in successor_node.predecessors:
            successor_node.predecessors.append(self)
    
    def to_dict(self):
        """Convert node to dictionary representation"""
        return {
            'id': self.id,
            'type': self.node_type,
            'statement': self.statement,
            'successors': [s.id for s in self.successors],
            'predecessors': [p.id for p in self.predecessors]
        }

class CFGGenerator:
    """Control Flow Graph Generator"""
    
    def __init__(self):
        self.graph = nx.DiGraph()  # Directed graph to represent control flow
        self.node_counter = 0
        self.entry_node = None
        self.exit_nodes = []
        self.nodes = {}
        
    def create_node(self, statement: str, node_type: str = "statement") -> CFGNode:
        """Create a new CFG node"""
        node = CFGNode(self.node_counter, statement, node_type)
        self.graph.add_node(self.node_counter, node=node)
        self.nodes[self.node_counter] = node
        self.node_counter += 1
        return node
    
    def generate_cfg(self, code: str) -> Dict[str, Any]:
        """Generate the CFG from Python code"""
        lines = code.split("\n")  # Split code into lines
        
        # Create the entry node
        self.entry_node = self.create_node("ENTRY", "entry")
        current_node = self.entry_node
        
        for line in lines:
            line = line.strip()
            if not line:
                continue
            
            if line.startswith("if") or line.startswith("else") or line.startswith("elif"):
                current_node = self.process_conditional(line, current_node)
            elif line.startswith("while") or line.startswith("for"):
                current_node = self.process_loop(line, current_node)
            else:
                current_node = self.process_statement(line, current_node)
        
        # After processing all lines, create an exit node if not already created
        if current_node:
            exit_node = self.create_node("EXIT", "exit")
            current_node.add_successor(exit_node)
            self.exit_nodes.append(exit_node)
        
        return self.create_cfg_structure()

    def process_statement(self, statement: str, current_node: CFGNode) -> CFGNode:
        """Process regular statements in the code"""
        new_node = self.create_node(statement, "statement")
        current_node.add_successor(new_node)
        return new_node
    
    def process_conditional(self, statement: str, current_node: CFGNode) -> CFGNode:
        """Process if-else conditionals"""
        condition_node = self.create_node(statement, "if_condition")
        current_node.add_successor(condition_node)
        
        # Implicitly create an else node if there’s no else part
        else_node = self.create_node("# Implicit else", "else_implicit")
        condition_node.add_successor(else_node)
        
        # Return the node after the conditional (could be a merge point)
        merge_node = self.create_node("# If-else merge", "merge")
        else_node.add_successor(merge_node)
        condition_node.add_successor(merge_node)
        
        return merge_node
    
    def process_loop(self, statement: str, current_node: CFGNode) -> CFGNode:
        """Process loops (for, while)"""
        loop_node = self.create_node(statement, "loop")
        current_node.add_successor(loop_node)
        
        # Loop back to the start of the loop
        loop_node.add_successor(loop_node)
        
        # Create an exit node for the loop
        exit_node = self.create_node("# Loop exit", "loop_exit")
        loop_node.add_successor(exit_node)
        
        return exit_node
    
    def create_cfg_structure(self) -> Dict[str, Any]:
        """Generate final CFG structure in the form of nodes and edges"""
        nodes_data = [node.to_dict() for node in self.nodes.values()]
        edges_data = [(node.id, successor.id) for node in self.nodes.values() for successor in node.successors]
        
        return {
            'nodes': nodes_data,
            'edges': edges_data,
            'entry_node': self.entry_node.id if self.entry_node else None,
            'exit_nodes': [node.id for node in self.exit_nodes],
            'node_count': len(self.nodes),
            'edge_count': len(edges_data)
        }

# Now we can create the generator, pass the code, and generate the CFG

def generate_cfg_for_code(code: str) -> Dict[str, Any]:
    """Generate CFG for the provided Python code"""
    cfg_generator = CFGGenerator()
    return cfg_generator.generate_cfg(code)




In [ ]:

def generate_cfg_for_code(code: str) -> Dict[str, Any]:
    """Generate CFG for the provided Python code"""
    cfg_generator = CFGGenerator()
    return cfg_generator.generate_cfg(code)


# cfg_generator = ImprovedCFGGenerator()

# def generate_cfg_for_code(code: str):
#     return cfg_generator.generate_cfg(code)

updated_df["cfg"] = updated_df["Code"].apply(generate_cfg_for_code)
print(updated_df["cfg"])


In [ ]:
updated_df.columns

In [ ]:
# # Build Prompts

# def build_prompt_template(code, metadata, cfg, ast_embeddings, mode="semantics"):
#     return f"""Generate a concise Python docstring for the following function:\n\n
# # Function name: {metadata["name"]} \n
# # Parameters: {metadata["params"]} \n
# # Return Type: {metadata["return_type"]} \n
# Control Flow Graph: {cfg} \n
# Abstract Syntax Tree Embeddings: {ast_embeddings}
# Code: \n{code}
# """
# updated_df["input_text"] = updated_df.apply(lambda row: build_prompt_template(row["Code"], row["metadata"], row['cfg'], row['encoder_output']), axis=1)
# updated_df["target_text"] = updated_df["Docstring"]   
# updated_df

# RAG


In [ ]:
# --- Add these imports at the top ---
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

# --- Build the retriever on your code corpus ---
corpus = updated_df["Code"].tolist()
vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5))
corpus_embeddings = vectorizer.fit_transform(corpus)
retriever = NearestNeighbors(n_neighbors=3, metric="cosine").fit(corpus_embeddings)

def retrieve_similar_code(query_code, top_k=3):
    query_vec = vectorizer.transform([query_code])
    distances, indices = retriever.kneighbors(query_vec, n_neighbors=top_k)
    return [corpus[i] for i in indices[0] if corpus[i] != query_code]

# --- Modify your prompt template to include retrieved context ---
def build_rag_cot_prompt(code, metadata, cfg, ast_embeddings, mode="semantics"):
    retrieved_snippets = retrieve_similar_code(code, top_k=3)
    retrieved_section = "\n\n".join([f"# Retrieved Example {i+1}:\n{snippet}" for i, snippet in enumerate(retrieved_snippets)])
    return f"""Generate a concise Python docstring for the following function.

First, think step by step about what the function does, its parameters, return type, and control flow. Use the retrieved examples for inspiration. Then, write the docstring.

# Function name: {metadata["name"]}
# Parameters: {metadata["params"]}
# Return Type: {metadata["return_type"]}
Control Flow Graph: {cfg}
Abstract Syntax Tree Embeddings: {ast_embeddings}
{retrieved_section}
Code:
{code}

# Step-by-step reasoning:
"""

# Update your DataFrame to use the new prompt
updated_df["input_text"] = updated_df.apply(
    lambda row: build_rag_cot_prompt(row["Code"], row["metadata"], row["cfg"], row["encoder_output"]), axis=1
)
updated_df["target_text"] = updated_df["Docstring"]   
updated_df

In [ ]:
dataset = Dataset.from_pandas(updated_df[["input_text", "target_text"]])
dataset = dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = dataset["train"]
test_dataset = dataset["test"]

# Tokenization


In [ ]:

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
# Baseline Model

model_name = "Salesforce/codet5-base"  #  

tokenizer = AutoTokenizer.from_pretrained(model_name)

def preprocess(examples):
    model_inputs = tokenizer(examples["input_text"], max_length=512, truncation=True)
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(examples["target_text"], max_length=128, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_dataset = train_dataset.map(preprocess, batched=True)
test_dataset = test_dataset.map(preprocess, batched=True)


# Load Model


In [ ]:
!pip list | findstr torch

In [ ]:
import torch

if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using MPS (Apple Silicon GPU) for training.")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using CUDA (NVIDIA GPU) for training.")
else:
    device = torch.device("cpu")
    print("CUDA and MPS not available, using CPU.")

model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

# Training Arguments


In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./codet5-finetuned-docstring",
    eval_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=3,
    predict_with_generate=True,
    fp16=False,
    bf16=True,
    logging_dir="./logs",
    logging_steps=50
)

In [ ]:
# Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer
)

In [ ]:
# For dynamic Padding
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True
)
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator
)
trainer.train()

In [ ]:
import torch
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from rouge import Rouge
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import nltk
try:
    nltk.download('punkt')
except:
    pass

# Load your trained model and tokenizer
model_path = "./codet5-finetuned-docstring/checkpoint-711"
model = AutoModelForSeq2SeqLM.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)

# Set model to evaluation mode
model.eval()

def generate_docstring_prediction(input_text, max_length=128, num_beams=4):
    """Generate docstring prediction for given input text"""
    inputs = tokenizer(
        input_text, 
        return_tensors="pt", 
        max_length=512, 
        truncation=True, 
        padding=True
    )
    
    with torch.no_grad():
        outputs = model.generate(
            inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=max_length,
            num_beams=num_beams,
            early_stopping=True,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )
    
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return generated_text

def evaluate_on_test_dataset(test_dataset, num_samples=None):
    """Evaluate model on your test dataset"""
    
    # Convert dataset to list for easier processing
    test_data = test_dataset
    if num_samples:
        test_data = test_dataset.select(range(min(num_samples, len(test_dataset))))
    
    predictions = []
    references = []
    input_texts = []
    
    print(f"Evaluating on {len(test_data)} samples...")
    
    for i in range(len(test_data)):
        if i % 10 == 0:
            print(f"Processing sample {i+1}/{len(test_data)}")
        
        # Get the original input text (before tokenization)
        input_text = test_data[i]["input_text"]
        target_text = test_data[i]["target_text"]
        
        # Generate prediction
        predicted_docstring = generate_docstring_prediction(input_text)
        
        predictions.append(predicted_docstring)
        references.append(target_text)
        input_texts.append(input_text)
    
    return predictions, references, input_texts

def calculate_evaluation_metrics(predictions, references):
    """Calculate ROUGE and BLEU metrics"""
    
    # Filter out empty predictions and references
    valid_pairs = [(p, r) for p, r in zip(predictions, references) 
                   if p.strip() and r.strip()]
    
    if not valid_pairs:
        print("No valid prediction-reference pairs found!")
        return {}
    
    valid_predictions, valid_references = zip(*valid_pairs)
    
    # ROUGE scores
    try:
        rouge = Rouge()
        rouge_scores = rouge.get_scores(list(valid_predictions), list(valid_references), avg=True)
        rouge_1 = rouge_scores['rouge-1']['f']
        rouge_2 = rouge_scores['rouge-2']['f']
        rouge_l = rouge_scores['rouge-l']['f']
    except Exception as e:
        print(f"Error calculating ROUGE scores: {e}")
        rouge_1 = rouge_2 = rouge_l = 0.0
    
    # BLEU scores
    smoothie = SmoothingFunction().method4
    bleu_scores = []
    
    for pred, ref in valid_pairs:
        pred_tokens = pred.split()
        ref_tokens = [ref.split()]  # BLEU expects list of reference lists
        
        try:
            bleu = sentence_bleu(ref_tokens, pred_tokens, smoothing_function=smoothie)
            bleu_scores.append(bleu)
        except:
            bleu_scores.append(0.0)
    
    avg_bleu = np.mean(bleu_scores) if bleu_scores else 0.0
    
    return {
        'rouge_1': rouge_1,
        'rouge_2': rouge_2,
        'rouge_l': rouge_l,
        'bleu': avg_bleu,
        'valid_samples': len(valid_pairs),
        'total_samples': len(predictions)
    }

def display_sample_predictions(input_texts, predictions, references, num_samples=5):
    """Display sample predictions with their inputs and targets"""
    
    print("\n" + "="*100)
    print("SAMPLE PREDICTIONS")
    print("="*100)
    
    for i in range(min(num_samples, len(predictions))):
        print(f"\n--- Sample {i+1} ---")
        
        # Extract function name and code from input text for cleaner display
        input_text = input_texts[i]
        lines = input_text.split('\n')
        
        function_name = ""
        code_section = ""
        for line in lines:
            if line.startswith("Function name:"):
                function_name = line.split(":", 1)[1].strip()
            elif line.startswith("Code:"):
                code_section = input_text[input_text.find("Code:") + 5:].strip()
                break
        
        print(f"Function Name: {function_name}")
        print(f"Code Preview: {code_section[:200]}{'...' if len(code_section) > 200 else ''}")
        
        print(f"\nActual Docstring:")
        print(f'"""{references[i]}"""')
        
        print(f"\nPredicted Docstring:")
        print(f'"""{predictions[i]}"""')
        
        # Calculate individual ROUGE score for this sample
        try:
            rouge = Rouge()
            if predictions[i].strip() and references[i].strip():
                individual_score = rouge.get_scores([predictions[i]], [references[i]])[0]
                print(f"\nROUGE-1 F1 for this sample: {individual_score['rouge-1']['f']:.3f}")
        except:
            print(f"\nCould not calculate ROUGE for this sample")
        
        print("-" * 80)

def analyze_prediction_quality(predictions, references):
    """Analyze the quality and characteristics of predictions"""
    
    print(f"\n{'='*60}")
    print("PREDICTION QUALITY ANALYSIS")
    print(f"{'='*60}")
    
    # Length analysis
    pred_lengths = [len(pred.split()) for pred in predictions]
    ref_lengths = [len(ref.split()) for ref in references]
    
    print(f"Average predicted docstring length: {np.mean(pred_lengths):.2f} words")
    print(f"Average actual docstring length: {np.mean(ref_lengths):.2f} words")
    print(f"Prediction length std: {np.std(pred_lengths):.2f}")
    print(f"Reference length std: {np.std(ref_lengths):.2f}")
    
    # Quality indicators
    empty_predictions = sum(1 for pred in predictions if len(pred.strip()) == 0)
    very_short_predictions = sum(1 for pred in predictions if len(pred.split()) < 3)
    very_long_predictions = sum(1 for pred in predictions if len(pred.split()) > 50)
    
    print(f"\nQuality Indicators:")
    print(f"Empty predictions: {empty_predictions} ({empty_predictions/len(predictions)*100:.1f}%)")
    print(f"Very short predictions (<3 words): {very_short_predictions} ({very_short_predictions/len(predictions)*100:.1f}%)")
    print(f"Very long predictions (>50 words): {very_long_predictions} ({very_long_predictions/len(predictions)*100:.1f}%)")
    
    # Content analysis
    generic_phrases = ["function", "method", "return", "parameter", "argument"]
    predictions_with_generic = sum(1 for pred in predictions 
                                 if any(phrase in pred.lower() for phrase in generic_phrases))
    
    print(f"Predictions containing generic terms: {predictions_with_generic} ({predictions_with_generic/len(predictions)*100:.1f}%)")

def test_custom_function(code, metadata):
    """Test the model on a custom function"""
    
    # Use your original prompt template function
    prompt = build_prompt_template(code, metadata)
    predicted_docstring = generate_docstring_prediction(prompt)
    
    print("\n" + "="*60)
    print("CUSTOM FUNCTION TEST")
    print("="*60)
    print("Input Code:")
    print(code)
    print(f"\nFunction Metadata:")
    print(f"Name: {metadata['name']}")
    print(f"Parameters: {metadata['params']}")
    print(f"Return Type: {metadata['return_type']}")
    print(f"\nGenerated Docstring:")
    print(f'"""{predicted_docstring}"""')
    
    return predicted_docstring

def save_evaluation_results(input_texts, predictions, references, metrics, filename="detailed_evaluation_results.csv"):
    """Save detailed evaluation results to CSV"""
    
    results_df = pd.DataFrame({
        'input_prompt': input_texts,
        'actual_docstring': references,
        'predicted_docstring': predictions,
        'prediction_length': [len(pred.split()) for pred in predictions],
        'reference_length': [len(ref.split()) for ref in references]
    })
    
    results_df.to_csv(filename, index=False)
    
    # Save metrics summary
    metrics_df = pd.DataFrame([metrics])
    metrics_df.to_csv(filename.replace('.csv', '_metrics.csv'), index=False)
    
    print(f"\nResults saved to '{filename}'")
    print(f"Metrics saved to '{filename.replace('.csv', '_metrics.csv')}'")

def run_complete_evaluation(test_dataset, num_samples=100):
    """Run the complete evaluation pipeline"""
    
    print("="*60)
    print("STARTING MODEL EVALUATION")
    print("="*60)
    print(f"Total test samples available: {len(test_dataset)}")
    print(f"Evaluating on: {num_samples if num_samples else len(test_dataset)} samples")
    
    # Generate predictions
    predictions, references, input_texts = evaluate_on_test_dataset(test_dataset, num_samples)
    
    # Calculate metrics
    print("\nCalculating evaluation metrics...")
    metrics = calculate_evaluation_metrics(predictions, references)
    
    # Display results
    print(f"\n{'='*60}")
    print("EVALUATION RESULTS")
    print(f"{'='*60}")
    print(f"ROUGE-1 F1: {metrics.get('rouge_1', 0):.4f}")
    print(f"ROUGE-2 F1: {metrics.get('rouge_2', 0):.4f}")
    print(f"ROUGE-L F1: {metrics.get('rouge_l', 0):.4f}")
    print(f"BLEU Score: {metrics.get('bleu', 0):.4f}")
    print(f"Valid samples: {metrics.get('valid_samples', 0)}/{metrics.get('total_samples', 0)}")
    
    # Show sample outputs
    display_sample_predictions(input_texts, predictions, references, num_samples=5)
    
    # Quality analysis
    analyze_prediction_quality(predictions, references)
    
    # Save results
    save_evaluation_results(input_texts, predictions, references, metrics)
    
    return predictions, references, metrics

# Main execution
if __name__ == "__main__":
    print("Loading test dataset...")
    
    # Run evaluation (adjust num_samples as needed)
    predictions, references, metrics = run_complete_evaluation(test_dataset, num_samples=50)
    
    # Test on a custom function using your metadata structure
    print("\n" + "="*60)
    print("TESTING CUSTOM FUNCTION")
    print("="*60)
    
    sample_code = """def fibonacci(n):
    '''Calculate fibonacci number'''
    if n <= 1:
        return n
    return fibonacci(n-1) + fibonacci(n-2)"""
    
    sample_metadata = {
        "name": "fibonacci",
        "params": ["n"],
        "return_type": None
    }
    
    custom_prediction = test_custom_function(sample_code, sample_metadata)
    
    print(f"\n{'='*60}")
    print("EVALUATION COMPLETE")
    print(f"{'='*60}")
    print("Check the generated CSV files for detailed results!")  # SIDE Score

In [ ]:
# docstring partitioning
with open("output.txt", 'w', encoding='utf-8') as output_file:
    for each_row in df["version_data"]:
        commit_time = ""
        for each_version in each_row:
            if commit_time:
                if commit_time < each_version["commit_date_time"]:
                    commit_time = each_version["commit_date_time"]
            else:
                commit_time = each_version["commit_date_time"]
        
        introduction = "" 

        return_type_list = ["list", "tuple", "dict", "set", "frozenset", "int", "str", "float", "bool", None]
        return_value = ""
        counter = 0
        
        
        for each_version in each_row:
            if each_version["commit_date_time"] == commit_time:
                metadata_info = {
                    "code": "",
                    "parameter": ""
                }
                dataset_code = each_version["code"]
                parameters = dataset_code[dataset_code.find("(") : dataset_code.find(")") + 1]
                output_file.write("-------------------\n")
                output_file.write(dataset_code)
                output_file.write("\n")
                output_file.write(parameters)
                output_file.write("\n")
                metadata_info["code"] = dataset_code
                metadata_info["parameter"] = parameters
    
    # for each_version in each_row:
    #     counter += 1
    #     print(f"Row Number: {counter}")
    #     if each_version["commit_date_time"] == commit_time:
    #         code = each_version["code"]
            
    #         # generate abstract syntax tree
    #         abstract_syntax_tree = ast.parse(code)
    #         print(ast.dump(abstract_syntax_tree, indent=4))
            
            